# 🚀 ComfyUI + MiniMax-H3 on Google Colab (A100 GPU Edition)

Google AI Pro 等のプランで付与される **Colab Compute Units (CU)** を活用して、強力な最新動画生成モデル **MiniMax-H3 (Hailuo)** を A100 (40GB VRAM) 環境で動かすためのオールインワン検証テンプレートです。

> 📖 **詳細な解説・検証記事 (Zenn)**: [Google AI Pro(2,900円)に課金するとA100 40GBがColab経由で毎月37時間分使える話](https://zenn.dev/grand2/articles/ddba80ba400f6f)

### 💡 特徴・ポイント
- **A100 GPU 最適化**: MiniMax-H3 の大規模モデル (`int8_convrot` 等) + Text Encoder + Audio/Video VAE をストレスなくロード。
- **🚀 Turbo LoRA 最適化 (推奨)**: 通常 25 ステップ（約30分）かかる生成を、**わずか 6〜8 ステップ（約7分・所要時間 1/4）へ激変** させる Turbo LoRA を自動配備。
- **高速化スタック対応**: `SageAttention`、`TeaCache` などの最新高速化ノード群をトグル1つで自動セットアップ。
- **Google Drive 完全永続化 (モデルキャッシュ & 出力動画)**:
  - モデル・LoRA（約20GB）を `MyDrive/ComfyUI_Models/` にキャッシュし、2回目以降のDLをスキップ（起動待ち0秒）。
  - 生成された動画・画像は **`MyDrive/ComfyUI_Outputs/` に自動保存**。ランタイムが切断・終了しても成果物が消えません。
- **Cloudflare Tunnel (無料・トークン不要)**: ngrok 不要ですぐにセキュアな一時公開 URL (`trycloudflare.com`) を自動発行。

> ⚠️ **注意**: ノートブック上部のメニュー「ランタイム」→「ランタイムのタイプを変更」から、**GPU (A100)** が選択されていることを確認してください。

## Step 1: 環境確認 & Google Drive マウント (永続化連携)

In [ ]:
# GPU および CUDA バージョンの確認 (A100 がアサインされているか確認)
!nvidia-smi

# モデルキャッシュ & 出力動画を Google Drive に永続保存
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully!")

## Step 2: ComfyUI 本体 & 高速化スタックのセットアップ

- **ENABLE_ACCELERATION (推奨)**: `SageAttention`、`TeaCache` などの最新高速化ノードを有効化。
- **CUDA 12.8 / 13.0 互換性ハンドリング**: PyTorch cu128 とホスト CUDA 13.0 の互換レイヤーを安全に処理します。

In [ ]:
import os

# @title 高速化オプション設定
ENABLE_ACCELERATION = True  # @param {type:"boolean"}
INSTALL_TEACACHE = True     # @param {type:"boolean"}

%cd /content

# ComfyUI 本体のクローン (最新版)
if not os.path.exists("/content/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
else:
    %cd /content/ComfyUI
    !git pull
    %cd /content

# 依存ライブラリのインストール
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q huggingface_hub

# SageAttention & Triton セットアップ
if ENABLE_ACCELERATION:
    print("⚡ 高速化スタック (SageAttention, Triton 等) をセットアップ中...")
    !pip install -q triton sageattention

# ComfyUI-Manager のインストール
%cd /content/ComfyUI/custom_nodes
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-Manager"):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git

# MiniMax-H3 Easy ノード
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-Easy"):
    !git clone https://github.com/kijai/ComfyUI-MiniMaxH3-Easy.git || true

# TeaCache 高速化ノード
if INSTALL_TEACACHE and not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-TeaCache"):
    !git clone https://github.com/chengzeyi/ComfyUI-MiniMaxH3-TeaCache.git || true

%cd /content/ComfyUI
print("✅ ノードおよび依存ライブラリの準備が完了しました！")

## Step 3: MiniMax-H3 モデル & Turbo LoRA の準備 (Google Drive 連携)

- **Turbo LoRA (`DOWNLOAD_TURBO_LORA = True`)**: `lightx2v/Minimax-h3-Turbo` をダウンロードし、**8ステップサンプリング（生成時間1/4）** を可能にします。
- `USE_GOOGLE_DRIVE = True` の場合：
  - **モデル & LoRA キャッシュ**: `/content/drive/MyDrive/ComfyUI_Models/` (次回以降 DL スキップ)
  - **動画出力先**: `/content/drive/MyDrive/ComfyUI_Outputs/` (インスタンス終了後も成果物を保持)

In [ ]:
import os
from huggingface_hub import hf_hub_download

# @title Turbo LoRA ダウンロード設定
DOWNLOAD_TURBO_LORA = True  # @param {type:"boolean"}

# モデルおよび出力先のディレクトリ準備
if USE_GOOGLE_DRIVE:
    MODELS_BASE = "/content/drive/MyDrive/ComfyUI_Models"
    OUTPUT_DIR = "/content/drive/MyDrive/ComfyUI_Outputs"
    os.makedirs(MODELS_BASE, exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 1. 出力フォルダを Google Drive にシンボリックリンク
    local_output = "/content/ComfyUI/output"
    if os.path.exists(local_output) and not os.path.islink(local_output):
        !rm -rf {local_output}
    if not os.path.exists(local_output):
        os.symlink(OUTPUT_DIR, local_output)

    # 2. モデルフォルダ群を Google Drive にシンボリックリンク
    for sub in ["diffusion_models", "text_encoders", "vae", "loras"]:
        drive_sub = os.path.join(MODELS_BASE, sub)
        local_sub = os.path.join("/content/ComfyUI/models", sub)
        os.makedirs(drive_sub, exist_ok=True)
        if os.path.exists(local_sub) and not os.path.islink(local_sub):
            !rm -rf {local_sub}
        if not os.path.exists(local_sub):
            os.symlink(drive_sub, local_sub)
    print(f"📁 Google Drive キャッシュ & 出力先を連携しました:\n  - Models: {MODELS_BASE}\n  - Outputs: {OUTPUT_DIR}")
else:
    MODELS_BASE = "/content/ComfyUI/models"

# 1. MiniMax-H3 ベースモデル群
REPO_ID = "Comfy-Org/MiniMax-H3"
files_to_download = [
    ("diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", REPO_ID),
    ("text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", REPO_ID),
    ("vae/minimax_h3_video_vae_fp16.safetensors", REPO_ID),
    ("vae/minimax_h3_audio_vae_fp32.safetensors", REPO_ID)
]

# 2. Turbo LoRA (8-Step 蒸留)
if DOWNLOAD_TURBO_LORA:
    files_to_download.append((
        "loras/minimax_h3_fl2v_turbo_8step_v1.0_768p_bf16.safetensors",
        "lightx2v/Minimax-h3-Turbo"
    ))

print("📥 MiniMax-H3 モデル & Turbo LoRA の確認・ダウンロードを開始...")
for filename, repo in files_to_download:
    target_path = os.path.join(MODELS_BASE, filename)
    sub_folder = os.path.dirname(filename)
    file_name_only = os.path.basename(filename)
    if os.path.exists(target_path) and os.path.getsize(target_path) > 1000000:
        print(f"⚡ キャッシュ検出 (DLスキップ): {filename}")
    else:
        print(f"⬇️ ダウンロード中: {filename} (from {repo}) ...")
        hf_hub_download(
            repo_id=repo,
            filename=file_name_only if repo != REPO_ID else filename,
            local_dir=os.path.join(MODELS_BASE, sub_folder) if repo != REPO_ID else MODELS_BASE,
            local_dir_use_symlinks=False
        )

print("✅ すべてのモデルおよび Turbo LoRA の準備が完了しました！")

## Step 4: ComfyUI 起動 & Cloudflare Tunnel 経由でアクセス

バックグラウンドで ComfyUI を起動し、Cloudflare Tunnel (`trycloudflare.com`) の安全なパブリック URL を発行して**起動状態を維持**します。
表示された `https://xxxx.trycloudflare.com` のリンクをクリックすると ComfyUI の WebUI が開きます。

### 💡 Turbo LoRA を使った超高速ワークフローの組み方
1. ComfyUI を開いたら、モデルローダー（`UNETLoader`）の直後に **`LoraLoader`** を挟みます。
2. LoRA に **`minimax_h3_fl2v_turbo_8step_...safetensors`** を選択します。
3. **KSampler の steps を `25` → `8` に変更** します（Euler / normal 推奨）。
4. これで **約 6〜8 分で 1 本の美麗動画が完成** します！

In [ ]:
import subprocess
import threading
import time
import re

# Cloudflared のダウンロード
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

# ComfyUI をバックグラウンドで起動
print("⚡ Starting ComfyUI...")
%cd /content/ComfyUI
comfy_proc = subprocess.Popen([
    "python", "main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--highvram"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# ComfyUI の起動ログを別スレッドで監視
def log_comfy():
    for line in iter(comfy_proc.stdout.readline, ''):
        print("[ComfyUI]", line, end="")

threading.Thread(target=log_comfy, daemon=True).start()

# 少し待機して Cloudflared トンネルを起動
time.sleep(5)
print("🌐 Starting Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen([
    "/content/cloudflared", "tunnel",
    "--url", "http://127.0.0.1:8188"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url_found = False
for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match and not url_found:
        url = match.group(0)
        print("\n" + "="*60)
        print(f"🎉 ComfyUI is LIVE: {url}")
        print("="*60 + "\n")
        print("💡 サーバーを稼働維持しています。(終了したい場合は本セルの停止ボタンを押すか、上部メニューから切断してください)")
        url_found = True

# トンネルプロセスが生きている間はセルを終了させず維持
try:
    tunnel_proc.wait()
except KeyboardInterrupt:
    print("\n⏹️ サーバーを停止しました。")
    comfy_proc.terminate()
    tunnel_proc.terminate()

## 🛑 (作業完了時) 確実に課金をストップする方法

動画生成の検証が完了したら、**最も確実に Compute Units (CU) の消費をストップ** するため、以下の手順でインスタンスを破棄してください：

### 💡 【推奨・確実度 100%】Colab メニューから切断
1. Colab 画面上部メニューの **「ランタイム」** をクリック
2. **「ランタイムの接続を解除して削除」** を選択（確認画面で「はい」）

> 成果物（動画）は Google Drive（`MyDrive/ComfyUI_Outputs/`）にリアルタイム保存されているため、切断しても安全です。